# Stance Detection Model Training with Hyperparameter Tuning

This notebook demonstrates the process of training a stance detection model using a grid search for hyperparameter tuning. The goal is to find the optimal hyperparameters to maximize the performance of the model on a stance detection task.

## 1. Importing Libraries

We start by importing the necessary libraries and modules:
- `os`, `numpy`, and `pandas` for file operations, numerical computations, and data handling.
- `datasets`, `load_metric` from Hugging Face for dataset and metric handling.
- `train_test_split` from scikit-learn for splitting the data into training, validation, and test sets.
- `AutoTokenizer`, `AutoModelForSequenceClassification`, `Trainer`, `TrainingArguments`, `EarlyStoppingCallback` from Hugging Face's Transformers library for tokenization, model setup, training, and evaluation.
- `tqdm` for displaying a progress bar during the grid search process.

In [2]:
import os
import numpy as np
import pandas as pd
from datasets import Dataset, load_metric
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from tqdm.notebook import tqdm

c:\Users\Karine\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Setting Parameters

We define the parameters and hyperparameters for the training process:
- `SEED`: A random seed for reproducibility.
- `MODEL_NAME`: The pre-trained model to use.
- `EPOCHS`: Number of training epochs.
- `TRAIN_SIZE`, `VALIDATION_SIZE`, `TEST_SIZE`: Proportions of the dataset for training, validation, and testing.
- `DIMENSION_SIZE`: Maximum length for tokenization.
- Lists of possible values for batch size, learning rate, dropout rate, and weight decay to perform grid search.

In [7]:
# Parameters
SEED = 42
MODEL_NAME = "bert-base-multilingual-uncased"
EPOCHS = 10
TRAIN_SIZE = 0.8  # Percentage of data for training
VALIDATION_SIZE = 0.1  # Percentage of data for validation
TEST_SIZE = 0.1  # Percentage of data for testing
DIMENSION_SIZE = 512  # Maximum tokenization length

# Lists of hyperparameters to be tested in the grid search
list_batch_size = [4]
list_learning_rate = [3e-5, 1e-4, 2e-5, 4e-5, 5e-5]
list_dropout = [0.1, 0.2, 0.3, 0.4, 0.5]
list_weight_decay = [0.01, 0.1, 1]

# Set the random seed for reproducibility
np.random.seed(SEED)

## 3. Initializing Results Storage

We initialize a dictionary to store the results of each training phase and define the file paths for storing results.


In [4]:
# Dictionary to store the results of each training phase
results = {
    "Phase": [],
    "Epoch": [],
    "Metric": [],
    "Value": [],
    "Batch_Size": [],
    "Learning_Rate": [],
    "Dropout": [],
    "Weight_Decay": []
}

# File paths for storing results
results_file_path = "./detailed_metrics_by_epoch.csv"
output_dir = './results'
file_path_final_label = 'FinalLabels.csv'

## 4. Defining Helper Functions

### 4.1 Compute Metrics

A function to compute evaluation metrics such as accuracy, precision, recall, and F1-score for each class.


In [5]:
# Function to compute evaluation metrics
def compute_metrics(eval_pred):
    accuracy = load_metric("accuracy", trust_remote_code=True)
    precision = load_metric("precision", trust_remote_code=True)
    recall = load_metric("recall", trust_remote_code=True)
    f1 = load_metric("f1", trust_remote_code=True)
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision_scores = precision.compute(predictions=predictions, references=labels, average=None)
    recall_scores = recall.compute(predictions=predictions, references=labels, average=None)
    f1_scores = f1.compute(predictions=predictions, references=labels, average=None)

    metrics_per_class = {f"precision_class_{i}": p for i, p in enumerate(precision_scores["precision"])}
    metrics_per_class.update({f"recall_class_{i}": r for i, r in enumerate(recall_scores["recall"])})
    metrics_per_class.update({f"f1_class_{i}": f for i, f in enumerate(f1_scores["f1"])})

    return metrics_per_class

### 4.2 Prepare Data

A function to load and preprocess the dataset, split it into training, validation, and test sets, and return them as Hugging Face datasets.


In [6]:
# Function to prepare the data
def prepare_data(file_path, train_size, validation_size, test_size):
    df = pd.read_csv(file_path, usecols=['comment', 'label'])
    df['comment'] = df['comment'].astype(str)
    df['comment'] = df['comment'].apply(lambda x: x.replace('\n', ' ').replace('\t', ' ').strip())
    
    # Mapping the labels C, I, F to 0, 1, 2
    label_mapping = {'C': 0, 'I': 1, 'F': 2}
    df['label'] = df['label'].map(label_mapping)
    
    df = df.sample(frac=1).reset_index(drop=True)

    print(df.groupby("label").count())

    train_val_df, test_df = train_test_split(df, test_size=test_size, stratify=df['label'], random_state=SEED)
    train_df, val_df = train_test_split(train_val_df, test_size=validation_size/(train_size + validation_size), stratify=train_val_df['label'], random_state=SEED)

    train_dataset = Dataset.from_pandas(train_df)
    val_dataset = Dataset.from_pandas(val_df)
    test_dataset = Dataset.from_pandas(test_df)

    return train_dataset, val_dataset, test_dataset

### 4.3 Tokenize and Format

A function to tokenize the text data and format it for model training.

In [ ]:
# Function to tokenize and format the dataset
def tokenize_and_format(examples):
    tokenized_inputs = tokenizer(
        examples['comment'],
        padding='max_length',  # Ensure all sequences are padded to the max length
        max_length=DIMENSION_SIZE,
        truncation=True  # Ensure all sequences are truncated to the max length
    )
    tokenized_inputs['labels'] = [int(label) for label in examples['label']]
    return tokenized_inputs

### 4.4 Custom Trainer

A custom Trainer class to log metrics during training and evaluation.


In [ ]:
# Custom Trainer class to log metrics during training and evaluation
class CustomTrainer(Trainer):
    def __init__(self, *args, combination=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.combination = combination

    def log_metrics(self, phase, epoch, metrics):
        for key, value in metrics.items():
            results["Phase"].append(phase)
            results["Epoch"].append(epoch)
            results["Metric"].append(key)
            results["Value"].append(value)
            results["Batch_Size"].append(self.combination["Batch_Size"])
            results["Learning_Rate"].append(self.combination["Learning_Rate"])
            results["Dropout"].append(self.combination["Dropout"])
            results["Weight_Decay"].append(self.combination["Weight_Decay"])

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix: str = "eval"):
        output = super().evaluate(eval_dataset, ignore_keys, metric_key_prefix)
        phase = "Test" if metric_key_prefix == "test" else "Validation"
        epoch = self.state.epoch if hasattr(self.state, "epoch") else "Final"
        self.log_metrics(phase, epoch, output)

        if metric_key_prefix != "test":
            train_output = super().evaluate(self.train_dataset, ignore_keys, "train")
            self.log_metrics("Training", epoch, train_output)

        return output

    

## 5. Loading Previous Results

If previous results exist, we load them to continue the grid search from where it left off.


In [ ]:
# Load previous results if they exist
if os.path.exists(results_file_path):
    results_df = pd.read_csv(results_file_path)
    processed_combinations = results_df[['Batch_Size', 'Learning_Rate', 'Dropout', 'Weight_Decay']].drop_duplicates().to_dict('records')
else:
    results_df = pd.DataFrame()
    processed_combinations = []

# Calculate total and processed combinations
total_combinations = len(list_batch_size) * len(list_learning_rate) * len(list_dropout) * len(list_weight_decay)
processed_count = len(processed_combinations)

## 6. Grid Search for Hyperparameter Tuning

We perform a grid search to find the best hyperparameters by training models with different combinations of hyperparameters and evaluating their performance.


In [ ]:
# Loop to vary hyperparameters with TQDM progress bar
with tqdm(total=total_combinations, initial=processed_count) as pbar:
    for BATCH_SIZE in list_batch_size:
        for LEARNING_RATE in list_learning_rate:
            for DROPOUT in list_dropout:
                for WEIGHT_DECAY in list_weight_decay:
                    current_combination = {
                        "Batch_Size": BATCH_SIZE,
                        "Learning_Rate": LEARNING_RATE,
                        "Dropout": DROPOUT,
                        "Weight_Decay": WEIGHT_DECAY
                    }
                    print(f"BATCH_SIZE: {BATCH_SIZE}\nLEARNING_RATE: {LEARNING_RATE}\nDROP_OUT: {DROPOUT}\nWEIGHT_DECAY: {WEIGHT_DECAY}")
                    
                    # Skip already processed combinations
                    if current_combination in processed_combinations:
                        pbar.update(1)
                        continue

                    # Load tokenizer and model
                    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
                    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3, hidden_dropout_prob=DROPOUT, attention_probs_dropout_prob=DROPOUT)

                    # Training arguments setup
                    training_args = TrainingArguments(
                        output_dir=output_dir,
                        num_train_epochs=EPOCHS,
                        per_device_train_batch_size=BATCH_SIZE,
                        per_device_eval_batch_size=BATCH_SIZE,
                        warmup_steps=500,
                        weight_decay=WEIGHT_DECAY,
                        logging_dir='./logs',
                        learning_rate=LEARNING_RATE,
                        evaluation_strategy="epoch",
                        save_strategy="epoch",
                        save_total_limit=1,
                        load_best_model_at_end=True,  # Load the best model at the end of training
                        metric_for_best_model="eval_loss",  # Metric to use for early stopping
                        greater_is_better=False,  # Whether the metric should be maximized
                    )

                    # Prepare and tokenize datasets
                    train_dataset, val_dataset, test_dataset = prepare_data(file_path_final_label, TRAIN_SIZE, VALIDATION_SIZE, TEST_SIZE)
                    train_dataset = train_dataset.map(tokenize_and_format, batched=True)
                    val_dataset = val_dataset.map(tokenize_and_format, batched=True)
                    # test_dataset = test_dataset.map(tokenize_and_format, batched=True)

                    # Initialize custom trainer
                    trainer = CustomTrainer(
                        combination=current_combination,
                        model=model,
                        args=training_args,
                        train_dataset=train_dataset,
                        eval_dataset=val_dataset,
                        compute_metrics=compute_metrics,
                        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # Adding Early Stopping Callback
                    )

                    # Train the model
                    trainer.train()
                    # trainer.evaluate(test_dataset, metric_key_prefix="test")

                    # Save and combine results
                    new_results_df = pd.DataFrame(results)
                    combined_results_df = pd.concat([results_df, new_results_df]).drop_duplicates().reset_index(drop=True)
                    combined_results_df.to_csv(results_file_path, index=False)

                    pbar.update(1)

# Stance Detection Model Training with Best Hyperparameters

This notebook demonstrates the process of training a stance detection model using the best hyperparameters identified from a previous grid search. The goal is to train the final model and evaluate its performance on the test set.

## 1. Importing Libraries

We start by importing the necessary libraries and modules:
- `os`, `numpy`, and `pandas` for file operations, numerical computations, and data handling.
- `datasets`, `load_metric` from Hugging Face for dataset and metric handling.
- `train_test_split` from scikit-learn for splitting the data into training and test sets.
- `AutoTokenizer`, `AutoModelForSequenceClassification`, `Trainer`, `TrainingArguments` from Hugging Face's Transformers library for tokenization, model setup, and training.


In [18]:
import os
import numpy as np
import pandas as pd
from datasets import Dataset, load_metric
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback

## 2. Setting Parameters

We define the parameters for the training process:
- `SEED`: A random seed for reproducibility.
- `MODEL_NAME`: The pre-trained model to use.
- `TEST_SIZE`: Proportion of the dataset for testing.


In [18]:
# Parameters
SEED = 42
MODEL_NAME = "bert-base-multilingual-uncased"

TEST_SIZE = 0.1  # Percentage of data for testing
DIMENSION_SIZE = 512  # Maximum tokenization length


## 3. Defining Helper Functions

### 3.1 Compute Metrics

A function to compute evaluation metrics such as accuracy, precision, recall, and F1-score for each class.

In [18]:
# Function to compute performance metrics
def compute_metrics(eval_pred):
    accuracy = load_metric("accuracy", trust_remote_code=True)
    precision = load_metric("precision", trust_remote_code=True)
    recall = load_metric("recall", trust_remote_code=True)
    f1 = load_metric("f1", trust_remote_code=True)
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision_scores = precision.compute(predictions=predictions, references=labels, average=None)
    recall_scores = recall.compute(predictions=predictions, references=labels, average=None)
    f1_scores = f1.compute(predictions=predictions, references=labels, average=None)

    metrics_per_class = {f"precision_class_{i}": p for i, p in enumerate(precision_scores["precision"])}
    metrics_per_class.update({f"recall_class_{i}": r for i, r in enumerate(recall_scores["recall"])})
    metrics_per_class.update({f"f1_class_{i}": f for i, f in enumerate(f1_scores["f1"])})

    return metrics_per_class

### 3.2 Prepare Data

A function to load and preprocess the dataset, split it into training and test sets, and return them as Hugging Face datasets.

In [18]:
# Function to prepare data
def prepare_data(file_path, test_size):
    df = pd.read_csv(file_path, usecols=['comment', 'label'])
    df['comment'] = df['comment'].astype(str)
    df['comment'] = df['comment'].apply(lambda x: x.replace('\n', ' ').replace('\t', ' ').strip())
    
    # Mapping the labels C, I, F to 0, 1, 2
    label_mapping = {'C': 0, 'I': 1, 'F': 2}
    df['label'] = df['label'].map(label_mapping)
    
    df = df.sample(frac=1).reset_index(drop=True)

    print(df.groupby("label").count())

    train_val_df, test_df = train_test_split(df, test_size=test_size, stratify=df['label'], random_state=SEED)

    train_val_dataset = Dataset.from_pandas(train_val_df)
    test_dataset = Dataset.from_pandas(test_df)

    return train_val_dataset, test_dataset

### 3.3 Tokenize and Format

A function to tokenize the text data and format it for model training.


In [18]:
# Function to tokenize and format data
def tokenize_and_format(examples):
    tokenized_inputs = tokenizer(examples['comment'], padding='max_length', max_length=DIMENSION_SIZE, truncation=True)
    tokenized_inputs['labels'] = examples['label']
    return tokenized_inputs

## 4. Loading Best Hyperparameters

We read the results from the grid search and select the best hyperparameter combination based on the lowest validation loss.


In [21]:
# Read the results file from the grid search
results_df = pd.read_csv('./detailed_metrics_by_epoch.csv')

# Select the best hyperparameter combination based on the lowest validation loss
best_combination_row = results_df.loc[results_df['Metric'] == 'eval_loss'].sort_values(by='Value', ascending=True).iloc[0]
BATCH_SIZE = best_combination_row['BATCH_SIZE']
LEARNING_RATE = best_combination_row['LEARNING_RATE']
DROP_OUT = best_combination_row['DROP_OUT']
WEIGHT_DECAY = best_combination_row['WEIGHT_DECAY']
EPOCH = best_combination_row['Epoch']

# Filter all metrics associated with the best hyperparameter combination
best_combination = results_df[(results_df['BATCH_SIZE'] == BATCH_SIZE) &
                              (results_df['LEARNING_RATE'] == LEARNING_RATE) &
                              (results_df['DROP_OUT'] == DROP_OUT) &
                              (results_df['WEIGHT_DECAY'] == WEIGHT_DECAY) &
                              (results_df['Epoch'] == EPOCH)]

# Print the best parameters and all performance metrics
print(f"Best Parameters:\nBATCH_SIZE: {BATCH_SIZE}\nLEARNING_RATE: {LEARNING_RATE}\nDROP_OUT: {DROP_OUT}\nWEIGHT_DECAY: {WEIGHT_DECAY}\nEPOCH: {EPOCH}")
print("\nAll Performance Metrics of the Best Model:")
print(best_combination[['Phase', 'Epoch', 'Metric', 'Value', 'BATCH_SIZE']])

Best Parameters:
BATCH_SIZE: 4
LEARNING_RATE: 3e-05
DROP_OUT: 0.4
WEIGHT_DECAY: 0.01
EPOCH: 4.0

All Performance Metrics of the Best Model:
          Phase  Epoch                   Metric     Value  BATCH_SIZE
740  Validation    4.0                eval_loss  0.489927           4
741  Validation    4.0   eval_precision_class_0  0.000000           4
742  Validation    4.0   eval_precision_class_1  0.932584           4
743  Validation    4.0   eval_precision_class_2  0.656250           4
744  Validation    4.0      eval_recall_class_0  0.000000           4
745  Validation    4.0      eval_recall_class_1  0.813725           4
746  Validation    4.0      eval_recall_class_2  0.884211           4
747  Validation    4.0          eval_f1_class_0  0.000000           4
748  Validation    4.0          eval_f1_class_1  0.869110           4
749  Validation    4.0          eval_f1_class_2  0.753363           4
750    Training    4.0               train_loss  0.424874           4
751    Training    4

## 5. Configuring the Model and Tokenizer

We configure the tokenizer and model using the best hyperparameters identified.

In [ ]:
# Configure the model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3, hidden_dropout_prob=DROP_OUT, attention_probs_dropout_prob=DROP_OUT)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=int(EPOCH),
    per_device_train_batch_size=int(BATCH_SIZE),
    per_device_eval_batch_size=int(BATCH_SIZE),
    warmup_steps=500,
    weight_decay=float(WEIGHT_DECAY),
    logging_dir='./logs',
    learning_rate=float(LEARNING_RATE),
    evaluation_strategy="no",  # No evaluation during final training
    save_strategy="epoch",
    save_total_limit=1,
)

## 6. Preparing the Data

We prepare the data by loading it, splitting it into training and test sets, and tokenizing it.

In [ ]:
# Prepare the data
train_dataset, test_dataset = prepare_data('./FinalLabels.csv', TEST_SIZE)
train_dataset = train_dataset.map(tokenize_and_format, batched=True)
test_dataset = test_dataset.map(tokenize_and_format, batched=True)

## 7. Training the Model

We train the model using the best hyperparameters and evaluate its performance on the test set.

In [24]:
# Train the model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,  # Evaluating on test set to keep validation during training
    compute_metrics=compute_metrics,
)

trainer.train()

# Evaluate on the test set
test_metrics = trainer.predict(test_dataset)
predictions_test = np.argmax(test_metrics.predictions, axis=-1)
labels_test = np.array(test_dataset['labels'])
test_metrics = compute_metrics((test_metrics.predictions, labels_test))

print("Test Metrics:", test_metrics)

Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss
500,0.476800
1000,0.725600
1500,0.673100
2000,0.609900
2500,0.564100


/home/smartdata_ig/c.ferreira/.conda/envs/c.ferreira_ENV/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Test Metrics: {'precision_class_0': 0.0, 'precision_class_1': 0.9602272727272727, 'precision_class_2': 0.6870229007633588, 'recall_class_0': 0.0, 'recall_class_1': 0.8284313725490197, 'recall_class_2': 0.9473684210526315, 'f1_class_0': 0.0, 'f1_class_1': 0.8894736842105264, 'f1_class_2': 0.7964601769911505}


/home/smartdata_ig/c.ferreira/.conda/envs/c.ferreira_ENV/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


## 8. Saving the Model and Results

Finally, we save the trained model and the predictions along with the true labels for future use.

In [25]:
# Save the model
trainer.save_model('./final_model')

# Save predictions and labels for later use
np.savez('./test_predictions_and_labels.npz', predictions=predictions_test, labels=labels_test)

# Bootstrapping Confidence Intervals for Model Metrics

This notebook demonstrates the process of estimating confidence intervals for model performance metrics using bootstrapping. The goal is to assess the stability and reliability of the model's precision, recall, and F1-score by repeatedly resampling the test set and computing these metrics.

## Strategy

Bootstrapping is a statistical technique that involves repeatedly sampling from a dataset with replacement to estimate the sampling distribution of a statistic. In this notebook, we apply bootstrapping to the model's predictions and the true labels from the test set to calculate confidence intervals for precision, recall, and F1-score for each class.

### Steps:

1. **Set Parameters**: Define the number of bootstrap iterations and a random seed for reproducibility.
2. **Load Predictions and Labels**: Load the saved predictions and true labels from the test set.
3. **Bootstrap Function**: Implement a function to perform bootstrapping, calculate metrics for each resampled set, and store the results.
4. **Compute Confidence Intervals**: Calculate the mean, standard deviation, and 95% confidence intervals for each metric.
5. **Save and Display Results**: Save the computed confidence intervals to a CSV file and display the results.

## 1. Importing Libraries

We start by importing the necessary libraries and modules:
- `numpy` and `pandas` for numerical computations and data handling.
- `precision_score`, `recall_score`, `f1_score` from scikit-learn for metric calculations.
- `tqdm` for displaying a progress bar during the bootstrapping process.

In [26]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score
from tqdm.notebook import tqdm

## 2. Setting Parameters

We define the parameters for the bootstrapping process:
- `BOOTSTRAP_ITERATIONS`: Number of bootstrap iterations to perform.
- `BOOTSTRAP_SEED`: Random seed for reproducibility.


In [26]:
# Parameters for bootstrapping
BOOTSTRAP_ITERATIONS = 1000  # Number of bootstrap iterations
BOOTSTRAP_SEED = 2024  # Random seed for reproducibility

## 3. Defining the Bootstrapping Function

### 3.1 Bootstrapping Function

A function to perform bootstrapping on the test set. It resamples the test set with replacement, calculates precision, recall, and F1-score for each class, and stores the results.


In [26]:
# Function to perform bootstrapping on the test set
def bootstrap_test(predictions, labels, n_iterations=BOOTSTRAP_ITERATIONS):
    rng = np.random.default_rng(BOOTSTRAP_SEED)  # Initialize a random number generator
    idx = np.arange(labels.shape[0])  # Create an array of indices for the labels

    # Lists to store metric scores for each class
    precision_scores_class_0, recall_scores_class_0, f1_scores_class_0 = [], [], []
    precision_scores_class_1, recall_scores_class_1, f1_scores_class_1 = [], [], []
    precision_scores_class_2, recall_scores_class_2, f1_scores_class_2 = [], [], []

    # Perform bootstrapping
    for _ in tqdm(range(n_iterations), desc="Bootstrap Iterations"):
        pred_idx = rng.choice(idx, size=idx.shape[0], replace=True)  # Resample the indices with replacement
        pred_boot = predictions[pred_idx]  # Get the bootstrap sample of predictions
        labels_boot = labels[pred_idx]  # Get the bootstrap sample of labels

        # Calculate precision, recall, and F1-score for class 0
        precision_class_0 = precision_score(labels_boot, pred_boot, labels=[0], average=None, zero_division=0)[0]
        recall_class_0 = recall_score(labels_boot, pred_boot, labels=[0], average=None, zero_division=0)[0]
        f1_class_0 = f1_score(labels_boot, pred_boot, labels=[0], average=None, zero_division=0)[0]

        # Calculate precision, recall, and F1-score for class 1
        precision_class_1 = precision_score(labels_boot, pred_boot, labels=[1], average=None, zero_division=0)[0]
        recall_class_1 = recall_score(labels_boot, pred_boot, labels=[1], average=None, zero_division=0)[0]
        f1_class_1 = f1_score(labels_boot, pred_boot, labels=[1], average=None, zero_division=0)[0]

        # Calculate precision, recall, and F1-score for class 2
        precision_class_2 = precision_score(labels_boot, pred_boot, labels=[2], average=None, zero_division=0)[0]
        recall_class_2 = recall_score(labels_boot, pred_boot, labels=[2], average=None, zero_division=0)[0]
        f1_class_2 = f1_score(labels_boot, pred_boot, labels=[2], average=None, zero_division=0)[0]

        # Store the scores for each iteration
        precision_scores_class_0.append(precision_class_0)
        recall_scores_class_0.append(recall_class_0)
        f1_scores_class_0.append(f1_class_0)

        precision_scores_class_1.append(precision_class_1)
        recall_scores_class_1.append(recall_class_1)
        f1_scores_class_1.append(f1_class_1)

        precision_scores_class_2.append(precision_class_2)
        recall_scores_class_2.append(recall_class_2)
        f1_scores_class_2.append(f1_class_2)

    # Calculate mean, standard deviation, and confidence intervals for each metric
    ci_metrics = {}
    for metric_name, scores in zip(
        ["precision_class_0", "recall_class_0", "f1_class_0",
         "precision_class_1", "recall_class_1", "f1_class_1",
         "precision_class_2", "recall_class_2", "f1_class_2"],
        [precision_scores_class_0, recall_scores_class_0, f1_scores_class_0,
         precision_scores_class_1, recall_scores_class_1, f1_scores_class_1,
         precision_scores_class_2, recall_scores_class_2, f1_scores_class_2]
    ):
        ci_metrics[metric_name] = {
            "mean": np.mean(scores),
            "std": np.std(scores),
            "lower_95_ci": np.percentile(scores, 2.5),
            "upper_95_ci": np.percentile(scores, 97.5)
        }

    return ci_metrics

## 4. Load Predictions and Labels

We load the saved predictions and true labels from the test set.

In [26]:
# Load the saved predictions and labels
data = np.load('./test_predictions_and_labels.npz')
predictions_test = data['predictions']
labels_test = data['labels']

## 5. Perform Bootstrapping on the Test Set

We apply the bootstrapping function to the test set predictions and labels to estimate the confidence intervals for the metrics.

In [26]:
# Perform bootstrapping on the test set
ci_metrics = bootstrap_test(predictions_test, labels_test, n_iterations=BOOTSTRAP_ITERATIONS)

## 6. Save and Display Results

We save the computed confidence intervals to a CSV file and print the results.

In [34]:
# Save the confidence interval results
results = pd.DataFrame(ci_metrics).T
results.to_csv('./bootstrap_results.csv')

print("Mean, Standard Deviation, and Confidence Intervals of Metrics for each class:")
print(results)

Mean, Standard Deviation, and Confidence Intervals of Metrics for each class:
                       mean       std  lower_95_ci  upper_95_ci
precision_class_0  0.000000  0.000000     0.000000     0.000000
recall_class_0     0.000000  0.000000     0.000000     0.000000
f1_class_0         0.000000  0.000000     0.000000     0.000000
precision_class_1  0.959720  0.014694     0.928562     0.988097
recall_class_1     0.828027  0.026103     0.772705     0.875664
f1_class_1         0.888762  0.016700     0.852031     0.918210
precision_class_2  0.686692  0.040870     0.603297     0.765629
recall_class_2     0.946795  0.023077     0.898990     0.988239
f1_class_2         0.795278  0.029509     0.735039     0.849791


# Text Classification Using Pre-Trained Model

This notebook demonstrates the process of loading a pre-trained text classification model, using it to classify new sentences, and mapping the model's predictions to human-readable labels. The goal is to utilize a fine-tuned BERT model to classify sentences into predefined categories.

## Strategy

The strategy involves the following steps:
1. **Load Pre-Trained Model**: Load a previously trained and saved model along with its tokenizer.
2. **Define Labels**: Create a mapping from numerical labels to human-readable labels.
3. **Input Sentences**: Prepare a list of example sentences to be classified.
4. **Classify Sentences**: Use the model to classify the sentences and map the predictions to the defined labels.
5. **Display Results**: Print each sentence along with its predicted classification.

## 1. Importing Libraries

We start by importing the necessary libraries and modules:
- `os` and `numpy` for file operations and numerical computations.
- `AutoTokenizer` and `AutoModelForSequenceClassification` from Hugging Face's Transformers library for tokenization and model loading.

In [28]:
import os
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

## 2. Load Pre-Trained Model

We load the tokenizer and the model using the model path where the fine-tuned model is saved. We also specify the original model name to ensure the correct tokenizer is loaded.

In [28]:
# Load the saved model
model_path = 'final_model/'
original_model_name = "bert-base-multilingual-uncased"
tokenizer = AutoTokenizer.from_pretrained(original_model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

## 3. Define Label Mapping

We define a dictionary to map the numerical labels predicted by the model to human-readable labels.


In [28]:
# Define a mapping from numerical labels to human-readable labels
label_map = {
    0: "Contrary",
    1: "Inconclusive",
    2: "Favorable"
}

## 4. Input Sentences

We prepare a list of example sentences in French to be classified by the model.

In [32]:
# List of example sentences for classification
sentences = [
    "J'adore vapoter !",  # F (Favorable)
    "Vapoter est la meilleure chose au monde.",  # F (Favorable)
    "Cette saveur est merveilleuse.",  # F (Favorable)
    
    "Le vapoteur détruira vos poumons et votre cœur. C'est un danger et un risque pour la santé.",  # C (Contrary)
    "Les enfants en deviennent accros. C'est absurde.",  # C (Contrary)
    "Cela doit être interdit. C’est nocif pour la santé et crée une dépendance chez les enfants." # C (Contrary)
    
    "Cette marque est très mauvaise.",  # I or F (Inconclusive or Favorable)
    "Je vis au Brésil.",  # I (Inconclusive)
    "Je regarde un match de football."  # I (Inconclusive)
]

## 5. Classify Sentences

We define a function to tokenize the input sentences, pass them through the model, and obtain the predictions. The function then returns the predicted labels.


In [33]:
# Function to classify sentences
def classify_sentences(sentences, tokenizer, model):
    # Tokenize the input sentences
    inputs = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True, max_length=512)
    # Pass the tokenized inputs through the model to get outputs
    outputs = model(**inputs)
    # Get the predicted labels by finding the index of the highest logit
    predictions = np.argmax(outputs.logits.detach().numpy(), axis=-1)
    return predictions

#We call the function to classify the example sentences and store the predictions.
predictions = classify_sentences(sentences, tokenizer, model)

# Print sentences and their respective classes
for sentence, prediction in zip(sentences, predictions):
    print(f"Sentence: {sentence}\nClassification: {label_map[prediction]}\n")

Sentence: J'adore vapoter !
Classification: Favorable

Sentence: Vapoter est la meilleure chose au monde.
Classification: Favorable

Sentence: Cette saveur est merveilleuse.
Classification: Inconclusive

Sentence: Le vapoteur détruira vos poumons et votre cœur. C'est un danger et un risque pour la santé.
Classification: Favorable

Sentence: Les enfants en deviennent accros. C'est absurde.
Classification: Inconclusive

Sentence: Cela doit être interdit. C’est nocif pour la santé et crée une dépendance chez les enfants.Cette marque est très mauvaise.
Classification: Inconclusive

Sentence: Je vis au Brésil.
Classification: Inconclusive

Sentence: Je regarde un match de football.
Classification: Inconclusive



# Using the Final Model to Classify the Whole Dataset

This notebook demonstrates how to use a pre-trained and fine-tuned model to classify a dataset and quantify the number of instances for each class. The results will be saved in a new column called `label_predicted`.

## Steps:

1. **Load Pre-Trained Model**: Load the previously trained model and tokenizer.
2. **Load Dataset**: Load the dataset that contains the comments to be classified.
3. **Classify Comments**: Use the model to classify the comments.
4. **Save Results**: Save the results with a new column `label_predicted` containing the predicted labels.

### Importing Libraries

We start by importing the necessary libraries and modules:
- `os` and `numpy` for file operations and numerical computations.
- `pandas` for data handling.
- `AutoTokenizer` and `AutoModelForSequenceClassification` from Hugging Face's Transformers library for tokenization and model loading.

In [46]:
import os
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.notebook import tqdm


### Load Pre-Trained Model

We load the tokenizer and the model using the model path where the fine-tuned model is saved. We also specify the original model name to ensure the correct tokenizer is loaded.

In [40]:
# Load the saved model
model_path = 'final_model/'
original_model_name = "bert-base-multilingual-uncased"
tokenizer = AutoTokenizer.from_pretrained(original_model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

### Load Dataset

We load the dataset that contains the comments to be classified.


In [63]:
# Load the dataset
df = pd.read_csv("cleaned_data/final_commments_fr_vape_match.csv")
df.head(2)

/tmp/ipykernel_481723/3590476735.py:2: DtypeWarning: Columns (9,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("cleaned_data/final_commments_fr_vape_match.csv")


,video_id,comment_id,author,author_profile_image_url,author_channel_url,author_channel_id,comment,published_at,updated_at,like_count,viewer_rating,can_rate,is_reply,parent_id,channel_id,language,language_langdetect
0,BaB4LEZFipU,UgxPR0_IDmNKSqtoqEF4AaABAg,@bouhschnou,https://yt3.ggpht.com/ytc/AIdro_nmrgOj8FjLrysO...,http://www.youtube.com/@bouhschnou,UCNDd8CfcrNHQvpxOHmgoJtw,@14:50 en faisant soi-même son mélange (nicoti...,2023-12-27 14:22:12+00:00,2023-12-27T14:22:12Z,0.0,none,True,False,NaN,UCp2kK3DyhpgFOdILF3BBC9A,fr,NaN
1,9UAKK7SPklc,Ugy2HM-9ljEgGGK-uGF4AaABAg,@maverickmike4459,https://yt3.ggpht.com/ytc/AIdro_lK6O2W2e3JHV4f...,http://www.youtube.com/@maverickmike4459,UC3XgmZT7yZoO2z3j34h0ZKQ,lol on parle d écologie sur une clope mais ...,2023-12-15 05:47:52+00:00,2023-12-15T05:47:52Z,0.0,none,True,False,NaN,UCYpRDnhk5H8h16jpS84uqsA,fr,NaN


### Classify Comments

We define a function to tokenize the input comments, pass them through the model, and obtain the predictions. The function then returns the predicted labels.

In [64]:
# Function to classify comments with progress bar
def classify_comments(comments, tokenizer, model):
    predictions = []
    for comment in tqdm(comments, desc="Classifying"):
        inputs = tokenizer(comment, return_tensors="pt", padding=True, truncation=True, max_length=512)
        outputs = model(**inputs)
        prediction = np.argmax(outputs.logits.detach().numpy(), axis=-1)
        predictions.append(prediction[0])
    return predictions

### We apply the function to classify the comments in the dataset and store the predictions in a new column `label_predicted`.

In [65]:
# Classify the comments in the dataset
df['label_predicted'] = classify_comments(df['comment'].tolist(), tokenizer, model)

Classifying:   0%|          | 0/368313 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter serve

### Save Results

We save the updated DataFrame with the new column `label_predicted` to a CSV file without the index.


In [ ]:
# Save the updated DataFrame with the new column 'label_predicted'
df.to_csv("cleaned_data/final_commments_fr_vape_match_with_predictions.csv", index=False)